# Cell 1 — configuration + model loading

In [ ]:
from pathlib import Path
import ast
import time

import numpy as np
import pandas as pd
import keras

try:
    from IPython.display import display
except Exception:
    display = print


# -----------------
# FILE PATHS
# -----------------

# Reference/labeled data.
# This is already mostly preprocessed.
# We use this only for drift reference and optional checking.
REFERENCE_CSV = Path("all_users_data_labeled.csv")

# Raw production/unlabeled Garmin files.
PROD_ACCEL_CSV = Path("2025-08-12-12-09-08_accelerometer.csv")
PROD_GYRO_CSV = Path("2025-08-12-12-09-08_gyroscope.csv")

# Keras model.
MODEL_PATH = Path("fine_tuned_model_1dcnnbilstm.keras")

# Output folder.
OUT_DIR = Path("inference_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_PRED_CSV = OUT_DIR / "production_window_predictions.csv"
ROW_PRED_CSV = OUT_DIR / "production_row_predictions.csv"
ACTIVITY_CONFIDENCE_CSV = OUT_DIR / "activity_confidence_summary.csv"
CLASS_PROB_SUMMARY_CSV = OUT_DIR / "class_probability_summary.csv"
FEATURE_DRIFT_CSV = OUT_DIR / "feature_drift_report.csv"
ACTIVITY_DRIFT_CSV = OUT_DIR / "activity_distribution_drift.csv"
DRIFT_SUMMARY_CSV = OUT_DIR / "drift_summary.csv"


# -----------------
# MODEL / PREPROCESSING SETTINGS
# -----------------

TARGET_HZ = 50
RESAMPLE_MS = 20

# Must match processing.ipynb
MERGE_TOLERANCE_MS = 1

# Model input shape = (batch, 200, 6)
WINDOW_SIZE = 200
STRIDE_SIZE = 100

WINDOW_SECONDS = WINDOW_SIZE / TARGET_HZ
STRIDE_SECONDS = STRIDE_SIZE / TARGET_HZ

FEATURE_COLS = [
    "Ax_w", "Ay_w", "Az_w",
    "Gx_w", "Gy_w", "Gz_w",
]

LABEL_CANDIDATES = ["activity", "Activity", "label", "Label"]

# Production raw accelerometer unit.
# Based on your processing.ipynb: raw Garmin accelerometer is converted by 0.00981.
PROD_ACCEL_UNIT = "milli_g"  # options: "milli_g", "mps2"


# -----------------
# CLASS NAMES
# -----------------

CLASS_NAMES = [
    "ear_rubbing",
    "forehead_rubbing",
    "hair_pulling",
    "hand_scratching",
    "hand_tapping",
    "knuckles_cracking",
    "nail_biting",
    "nape_rubbing",
    "sitting",
    "smoking",
    "standing",
]


# -----------------
# CONFIDENCE SETTINGS
# -----------------

HIGH_CONF_THRESHOLD = 0.80
MEDIUM_CONF_THRESHOLD = 0.60
UNCERTAIN_THRESHOLD = 0.60


# -----------------
# DRIFT SETTINGS
# -----------------

DRIFT_PSI_THRESHOLD = 0.25
DRIFT_KS_THRESHOLD = 0.20
DRIFT_MEAN_SHIFT_Z_THRESHOLD = 2.0
DRIFT_STD_RATIO_LOW = 0.5
DRIFT_STD_RATIO_HIGH = 2.0
ACTIVITY_DIST_DRIFT_TV_THRESHOLD = 0.25


# -----------------
# LOAD MODEL
# -----------------

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

model = keras.models.load_model(MODEL_PATH, compile=False)

print("Model loaded.")
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)

expected_input_shape = (None, WINDOW_SIZE, len(FEATURE_COLS))

if model.input_shape != expected_input_shape:
    print("WARNING: model input shape mismatch.")
    print("Expected:", expected_input_shape)
    print("Actual:", model.input_shape)

if model.output_shape[-1] != len(CLASS_NAMES):
    print("WARNING: model output classes do not match CLASS_NAMES.")
    print("Model output:", model.output_shape[-1])
    print("CLASS_NAMES:", len(CLASS_NAMES))

print("Ready.")

Model loaded.
Input shape: (None, 200, 6)
Output shape: (None, 11)
Ready.


# Cell 2 — preprocessing helper functions

In [2]:
def find_label_col(df: pd.DataFrame):
    for c in LABEL_CANDIDATES:
        if c in df.columns:
            return c
    return None


def parse_timestamp_series(s: pd.Series) -> pd.Series:
    """
    Handles numeric Unix timestamps and datetime strings.
    """
    if pd.api.types.is_numeric_dtype(s):
        med = pd.to_numeric(s, errors="coerce").dropna().median()

        if pd.notna(med) and med > 1e12:
            return pd.to_datetime(s, unit="ms", origin="unix", errors="coerce")

        return pd.to_datetime(s, unit="s", origin="unix", errors="coerce")

    return pd.to_datetime(s, errors="coerce")


def _to_list_if_list_like(x):
    """
    Converts '[1, 2, 3]' style cells to Python lists.
    """
    if isinstance(x, list):
        return x

    if isinstance(x, str):
        txt = x.strip()
        if txt.startswith("[") and txt.endswith("]"):
            try:
                return ast.literal_eval(txt)
            except Exception:
                return x

    return x


def expand_list_columns_fast(df: pd.DataFrame, columns_to_expand: list) -> pd.DataFrame:
    """
    Expands Garmin rows where one row contains multiple sample values as lists.
    """
    cols = [c for c in columns_to_expand if c in df.columns]

    if not cols:
        return df.copy()

    out = df.copy()

    has_list_like = False

    for c in cols:
        out[c] = out[c].map(_to_list_if_list_like)

        if out[c].map(lambda v: isinstance(v, list)).any():
            has_list_like = True

    if not has_list_like:
        return out.reset_index(drop=True)

    for c in cols:
        out[c] = out[c].map(lambda v: v if isinstance(v, list) else [v])

    try:
        out = out.explode(cols, ignore_index=True)

    except ValueError:
        rows = []

        for _, row in out.iterrows():
            max_len = max(
                len(row[c]) if isinstance(row[c], list) else 1
                for c in cols
            )

            for i in range(max_len):
                new_row = row.copy()

                for c in cols:
                    v = row[c]

                    if isinstance(v, list):
                        new_row[c] = v[i] if i < len(v) else np.nan

                rows.append(new_row)

        out = pd.DataFrame(rows).reset_index(drop=True)

    return out.reset_index(drop=True)


def normalize_reference_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalizes already-preprocessed reference/labeled CSV.

    Important:
    - Does NOT downsample.
    - Does NOT convert accelerometer units again.
    """
    out = df.copy()

    rename_map = {
        "accel_x": "Ax_w",
        "accel_y": "Ay_w",
        "accel_z": "Az_w",
        "gyro_x": "Gx_w",
        "gyro_y": "Gy_w",
        "gyro_z": "Gz_w",

        "calibrated_accel_x": "Ax_w",
        "calibrated_accel_y": "Ay_w",
        "calibrated_accel_z": "Az_w",
        "calibrated_gyro_x": "Gx_w",
        "calibrated_gyro_y": "Gy_w",
        "calibrated_gyro_z": "Gz_w",

        "Activity": "activity",
        "Label": "activity",
        "label": "activity",
    }

    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})

    for axis in ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]:
        wrist_col = f"{axis}_w"
        pocket_col = f"{axis}_p"

        if wrist_col not in out.columns and pocket_col in out.columns:
            out[wrist_col] = out[pocket_col]

    if "timestamp" in out.columns:
        out["timestamp"] = parse_timestamp_series(out["timestamp"])
    else:
        out["timestamp"] = (
            pd.Timestamp("2000-01-01")
            + pd.to_timedelta(np.arange(len(out)) * RESAMPLE_MS, unit="ms")
        )

    label_col = find_label_col(out)

    keep = ["timestamp"] + FEATURE_COLS

    if label_col is not None:
        keep.append(label_col)

    group_cols = [
        "User", "user", "user_id", "participant", "participant_id",
        "session", "session_id", "file", "filename", "recording_id",
    ]

    for c in group_cols:
        if c in out.columns and c not in keep:
            keep.append(c)

    missing = [c for c in FEATURE_COLS if c not in out.columns]

    if missing:
        raise ValueError(f"Reference CSV missing required columns: {missing}")

    out = out[keep].copy()

    for c in FEATURE_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.dropna(subset=["timestamp"] + FEATURE_COLS)

    if label_col is not None and label_col != "activity":
        out = out.rename(columns={label_col: "activity"})

    out = out.sort_values("timestamp").reset_index(drop=True)

    return out


def parse_prod_sensor_csv(path: Path, sensor_prefix: str) -> pd.DataFrame:
    """
    Parses raw Garmin accelerometer or gyroscope production CSV.

    Production preprocessing:
    - expand list-like sample rows
    - timestamp = timestamp + timestamp_ms + sample_time_offset
    - accelerometer: convert milli-g to m/s²
    - gyroscope: no conversion
    """
    df = pd.read_csv(path)

    list_like_cols = [
        "sample_time_offset",

        "accel_x", "accel_y", "accel_z",
        "gyro_x", "gyro_y", "gyro_z",

        "calibrated_accel_x", "calibrated_accel_y", "calibrated_accel_z",
        "calibrated_gyro_x", "calibrated_gyro_y", "calibrated_gyro_z",
    ]

    df = expand_list_columns_fast(df, list_like_cols)

    if "timestamp" not in df.columns:
        raise ValueError(f"'timestamp' column not found in {path}")

    base_ts = parse_timestamp_series(df["timestamp"])

    timestamp_ms = (
        pd.to_numeric(df["timestamp_ms"], errors="coerce").fillna(0)
        if "timestamp_ms" in df.columns
        else 0
    )

    sample_offset = (
        pd.to_numeric(df["sample_time_offset"], errors="coerce").fillna(0)
        if "sample_time_offset" in df.columns
        else 0
    )

    precise_ts = base_ts + pd.to_timedelta(timestamp_ms + sample_offset, unit="ms")

    out = pd.DataFrame({"timestamp": precise_ts})

    if sensor_prefix == "accel":
        accel_options = [
            ("accel_x", "accel_y", "accel_z"),
            ("calibrated_accel_x", "calibrated_accel_y", "calibrated_accel_z"),
        ]

        selected = next(
            (cols for cols in accel_options if all(c in df.columns for c in cols)),
            None,
        )

        if selected is None:
            raise ValueError(f"No accelerometer columns found in {path}")

        out["Ax_w"] = pd.to_numeric(df[selected[0]], errors="coerce")
        out["Ay_w"] = pd.to_numeric(df[selected[1]], errors="coerce")
        out["Az_w"] = pd.to_numeric(df[selected[2]], errors="coerce")

        if PROD_ACCEL_UNIT == "milli_g":
            out[["Ax_w", "Ay_w", "Az_w"]] = (
                out[["Ax_w", "Ay_w", "Az_w"]] * 0.00981
            )
        elif PROD_ACCEL_UNIT == "mps2":
            pass
        else:
            raise ValueError("PROD_ACCEL_UNIT must be 'milli_g' or 'mps2'")

    elif sensor_prefix == "gyro":
        gyro_options = [
            ("gyro_x", "gyro_y", "gyro_z"),
            ("calibrated_gyro_x", "calibrated_gyro_y", "calibrated_gyro_z"),
        ]

        selected = next(
            (cols for cols in gyro_options if all(c in df.columns for c in cols)),
            None,
        )

        if selected is None:
            raise ValueError(f"No gyroscope columns found in {path}")

        out["Gx_w"] = pd.to_numeric(df[selected[0]], errors="coerce")
        out["Gy_w"] = pd.to_numeric(df[selected[1]], errors="coerce")
        out["Gz_w"] = pd.to_numeric(df[selected[2]], errors="coerce")

    else:
        raise ValueError("sensor_prefix must be 'accel' or 'gyro'")

    out = out.dropna(subset=["timestamp"])
    out = out.sort_values("timestamp").reset_index(drop=True)

    return out


def build_production_from_two_files(accel_path: Path, gyro_path: Path) -> pd.DataFrame:
    """
    Builds production dataframe from separate raw accel + gyro files.
    Uses 1 ms merge tolerance, matching processing.ipynb.
    """
    acc = parse_prod_sensor_csv(accel_path, sensor_prefix="accel")
    gyr = parse_prod_sensor_csv(gyro_path, sensor_prefix="gyro")

    merged = pd.merge_asof(
        acc.sort_values("timestamp"),
        gyr.sort_values("timestamp"),
        on="timestamp",
        direction="nearest",
        tolerance=pd.Timedelta(milliseconds=MERGE_TOLERANCE_MS),
    )

    before = len(merged)
    merged = merged.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    dropped = before - len(merged)

    print(f"Accel rows: {len(acc):,}")
    print(f"Gyro rows: {len(gyr):,}")
    print(f"Merged rows: {len(merged):,}")
    print(f"Dropped after 1 ms merge/dropna: {dropped:,}")

    return merged


def downsample_production(df: pd.DataFrame) -> pd.DataFrame:
    """
    Downsamples raw production data to 50 Hz / 20 ms.
    Do not use this for all_users_data_labeled.csv.
    """
    out = df.copy()

    out = out.dropna(subset=["timestamp"])
    out = out.sort_values("timestamp").set_index("timestamp")

    agg = {c: "mean" for c in FEATURE_COLS}

    out = (
        out.resample(f"{RESAMPLE_MS}ms")
        .agg(agg)
        .dropna(subset=FEATURE_COLS)
        .reset_index()
    )

    return out


def build_windows(df: pd.DataFrame):
    """
    Builds raw time-series windows for Keras model.

    Output:
        X.shape = (n_windows, 200, 6)
    """
    missing = [c for c in FEATURE_COLS if c not in df.columns]

    if missing:
        raise ValueError(f"Missing feature columns: {missing}")

    arr = df[FEATURE_COLS].to_numpy(dtype=np.float32)

    X = []
    y = []
    meta = []

    has_labels = "activity" in df.columns

    for start_idx in range(0, len(arr) - WINDOW_SIZE + 1, STRIDE_SIZE):
        end_idx = start_idx + WINDOW_SIZE

        X.append(arr[start_idx:end_idx])

        meta.append({
            "start_time": df.iloc[start_idx]["timestamp"],
            "end_time": df.iloc[end_idx - 1]["timestamp"],
            "start_idx": start_idx,
            "end_idx": end_idx - 1,
        })

        if has_labels:
            y.append(df.iloc[start_idx:end_idx]["activity"].mode().iloc[0])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y) if has_labels else None
    meta = pd.DataFrame(meta)

    return X, y, meta

# Cell 3 — segment-aware windowing

In [3]:
def add_segment_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates segment_id to prevent windows from crossing:
    - user/session/activity boundaries
    - large timestamp gaps
    """
    out = df.copy()

    group_cols = [
        c for c in [
            "User", "user", "user_id", "participant", "participant_id",
            "session", "session_id", "file", "filename", "recording_id",
        ]
        if c in out.columns
    ]

    sort_cols = group_cols + ["timestamp"]
    out = out.sort_values(sort_cols).reset_index(drop=True)

    new_segment = pd.Series(False, index=out.index)

    if len(out) > 0:
        new_segment.iloc[0] = True

    for c in group_cols:
        new_segment |= out[c].astype(str).ne(out[c].astype(str).shift())

    if "activity" in out.columns:
        new_segment |= out["activity"].astype(str).ne(out["activity"].astype(str).shift())

    time_gap = out["timestamp"].diff() > pd.Timedelta(milliseconds=RESAMPLE_MS * 3)
    new_segment |= time_gap.fillna(True)

    out["segment_id"] = new_segment.cumsum()

    return out


def build_windows_segment_aware(df: pd.DataFrame):
    """
    Builds windows without crossing segment boundaries.

    Important:
    start_idx and end_idx are GLOBAL indices into the input dataframe.
    """
    df_seg = add_segment_id(df)

    X_parts = []
    y_parts = []
    meta_parts = []

    has_labels = "activity" in df_seg.columns

    for segment_id, seg_df in df_seg.groupby("segment_id", sort=False):
        seg_offset = int(seg_df.index[0])
        seg_df_local = seg_df.reset_index(drop=True)

        if len(seg_df_local) < WINDOW_SIZE:
            continue

        X_seg, y_seg, meta_seg = build_windows(seg_df_local)

        if len(X_seg) == 0:
            continue

        meta_seg["start_idx"] = meta_seg["start_idx"] + seg_offset
        meta_seg["end_idx"] = meta_seg["end_idx"] + seg_offset
        meta_seg["segment_id"] = segment_id

        if has_labels:
            meta_seg["activity"] = y_seg

        X_parts.append(X_seg)
        meta_parts.append(meta_seg)

        if has_labels:
            y_parts.append(y_seg)

    if len(X_parts) == 0:
        X = np.empty((0, WINDOW_SIZE, len(FEATURE_COLS)), dtype=np.float32)
        y = np.array([]) if has_labels else None
        meta = pd.DataFrame()
        return X, y, meta

    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0) if has_labels else None
    meta = pd.concat(meta_parts, ignore_index=True)

    return X, y, meta

## Cell 4 — load reference data and production data

In [4]:
def prepare_reference_data(reference_path: Path):
    """
    Loads already-preprocessed labeled/reference data.

    No downsampling.
    No accelerometer conversion.
    """
    if not reference_path.exists():
        print(f"WARNING: Reference CSV not found: {reference_path}")

        X_ref = np.empty((0, WINDOW_SIZE, len(FEATURE_COLS)), dtype=np.float32)
        y_ref = np.array([])
        meta_ref = pd.DataFrame()

        return None, X_ref, y_ref, meta_ref

    t0 = time.perf_counter()

    reference_raw = pd.read_csv(reference_path)
    print("Loaded reference CSV:", reference_raw.shape)

    reference_df = normalize_reference_columns(reference_raw)
    print("Normalized reference data:", reference_df.shape)

    if "activity" in reference_df.columns:
        before = len(reference_df)

        reference_df = reference_df[
            reference_df["activity"].astype(str).str.lower().str.strip() != "unknown"
        ].copy()

        after = len(reference_df)
        print(f"Removed unknown labels: {before - after:,}")

    reference_ds = reference_df.dropna(
        subset=["timestamp"] + FEATURE_COLS
    ).reset_index(drop=True)

    X_ref, y_ref, meta_ref = build_windows_segment_aware(reference_ds)

    elapsed = time.perf_counter() - t0
    print(f"Reference preprocessing finished in {elapsed:.2f} seconds")

    return reference_ds, X_ref, y_ref, meta_ref


def prepare_production_data(accel_path: Path, gyro_path: Path):
    """
    Full production preprocessing:
    raw accel + raw gyro
    → expand list rows
    → precise timestamps
    → accel conversion
    → 1 ms merge
    → 50 Hz downsample
    → gap-aware windows
    """
    if not accel_path.exists():
        raise FileNotFoundError(f"Production accel CSV not found: {accel_path}")

    if not gyro_path.exists():
        raise FileNotFoundError(f"Production gyro CSV not found: {gyro_path}")

    t0 = time.perf_counter()

    prod_raw = build_production_from_two_files(accel_path, gyro_path)
    print("Production raw merged:", prod_raw.shape)

    prod_ds = downsample_production(prod_raw)
    print("Production downsampled:", prod_ds.shape)

    X_prod, _, meta_prod = build_windows_segment_aware(prod_ds)

    elapsed = time.perf_counter() - t0
    print(f"Production preprocessing finished in {elapsed:.2f} seconds")

    return prod_raw, prod_ds, X_prod, meta_prod


# Load reference
reference_ds, X_ref, y_ref, meta_ref = prepare_reference_data(REFERENCE_CSV)

# Load production
prod_raw, prod_ds, X_prod, meta_prod = prepare_production_data(
    PROD_ACCEL_CSV,
    PROD_GYRO_CSV,
)


# Validation
expected_window_shape = (WINDOW_SIZE, len(FEATURE_COLS))

print("\n========== SUMMARY ==========")

print("\nReference:")
print("Rows:", None if reference_ds is None else reference_ds.shape)
print("X_ref:", X_ref.shape)
print("y_ref:", None if y_ref is None else y_ref.shape)
print("meta_ref:", meta_ref.shape)

print("\nProduction:")
print("Raw rows:", prod_raw.shape)
print("Processed rows:", prod_ds.shape)
print("X_prod:", X_prod.shape)
print("meta_prod:", meta_prod.shape)

if X_ref.shape[0] > 0 and X_ref.shape[1:] != expected_window_shape:
    raise ValueError(f"Wrong X_ref shape: {X_ref.shape}")

if X_prod.shape[0] == 0:
    raise ValueError("No production windows created.")

if X_prod.shape[1:] != expected_window_shape:
    raise ValueError(f"Wrong X_prod shape: {X_prod.shape}")

print("\nPreprocessing complete.")

Loaded reference CSV: (385326, 9)
Normalized reference data: (385326, 9)
Removed unknown labels: 0
Reference preprocessing finished in 1.59 seconds
Accel rows: 243,800
Gyro rows: 243,800
Merged rows: 230,366
Dropped after 1 ms merge/dropna: 13,434
Production raw merged: (230366, 7)
Production downsampled: (121726, 7)
Production preprocessing finished in 1.05 seconds

========== SUMMARY ==========

Reference:
Rows: (385326, 9)
X_ref: (3449, 200, 6)
y_ref: (3449,)
meta_ref: (3449, 6)

Production:
Raw rows: (230366, 7)
Processed rows: (121726, 7)
X_prod: (1216, 200, 6)
meta_prod: (1216, 5)

Preprocessing complete.


# Cell 5 — model inference + confidence

In [5]:
print("Running model inference...")
print("X_prod:", X_prod.shape)

pred_probs = model.predict(X_prod, verbose=0, batch_size=256)
pred_probs = np.asarray(pred_probs)

if pred_probs.ndim == 1:
    pred_probs = pred_probs.reshape(-1, 1)

if pred_probs.shape[1] != len(CLASS_NAMES):
    raise ValueError(
        f"Model output classes = {pred_probs.shape[1]}, "
        f"but CLASS_NAMES = {len(CLASS_NAMES)}"
    )

# Ensure probability distribution
row_sums = pred_probs.sum(axis=1)

if not np.allclose(row_sums, 1.0, atol=1e-3):
    exp_probs = np.exp(pred_probs - np.max(pred_probs, axis=1, keepdims=True))
    pred_probs = exp_probs / exp_probs.sum(axis=1, keepdims=True)

classes = np.array(CLASS_NAMES)

sorted_idx = np.argsort(pred_probs, axis=1)

top1_idx = sorted_idx[:, -1]
top2_idx = sorted_idx[:, -2]

top1_conf = pred_probs[np.arange(len(pred_probs)), top1_idx]
top2_conf = pred_probs[np.arange(len(pred_probs)), top2_idx]

pred_label = classes[top1_idx]
top2_label = classes[top2_idx]


# Window-level output
window_out = meta_prod.copy()

window_out["predicted_activity"] = pred_label
window_out["confidence"] = top1_conf
window_out["top2_activity"] = top2_label
window_out["top2_confidence"] = top2_conf
window_out["top1_top2_margin"] = window_out["confidence"] - window_out["top2_confidence"]

window_out["confidence_band"] = np.where(
    window_out["confidence"] >= HIGH_CONF_THRESHOLD,
    "high",
    np.where(
        window_out["confidence"] >= MEDIUM_CONF_THRESHOLD,
        "medium",
        "low",
    ),
)

window_out["is_uncertain"] = window_out["confidence"] < UNCERTAIN_THRESHOLD

# Probability for every activity
for i, cls in enumerate(CLASS_NAMES):
    window_out[f"prob_{cls}"] = pred_probs[:, i]

window_out.to_csv(WINDOW_PRED_CSV, index=False)

print("Saved:", WINDOW_PRED_CSV)

print("\nPredicted activity counts:")
print(window_out["predicted_activity"].value_counts())

print("\nConfidence band counts:")
print(window_out["confidence_band"].value_counts())

print("\nUncertain window %:", round(100 * window_out["is_uncertain"].mean(), 2), "%")

window_out.head()

Running model inference...
X_prod: (1216, 200, 6)
Saved: inference_outputs\production_window_predictions.csv

Predicted activity counts:
predicted_activity
ear_rubbing         511
hand_tapping        478
smoking             202
hair_pulling         17
nail_biting           6
forehead_rubbing      1
hand_scratching       1
Name: count, dtype: int64

Confidence band counts:
confidence_band
high      608
medium    340
low       268
Name: count, dtype: int64

Uncertain window %: 22.04 %


,start_time,end_time,start_idx,end_idx,segment_id,predicted_activity,confidence,top2_activity,top2_confidence,top1_top2_margin,...,prob_forehead_rubbing,prob_hair_pulling,prob_hand_scratching,prob_hand_tapping,prob_knuckles_cracking,prob_nail_biting,prob_nape_rubbing,prob_sitting,prob_smoking,prob_standing
0,2025-08-12 22:41:37.240,2025-08-12 22:41:41.260,0,199,1,ear_rubbing,0.575845,hand_tapping,0.305295,0.270551,...,0.000027,0.053817,0.000044,0.305295,1.926699e-06,0.001615,0.000004,0.000287,0.062697,0.000364
1,2025-08-12 22:41:39.240,2025-08-12 22:41:43.260,100,299,1,ear_rubbing,0.814720,smoking,0.128177,0.686543,...,0.000050,0.021147,0.000008,0.033633,1.365962e-06,0.002216,0.000005,0.000025,0.128177,0.000018
2,2025-08-12 22:41:41.280,2025-08-12 22:41:45.260,200,399,1,ear_rubbing,0.826161,hand_tapping,0.117481,0.708680,...,0.001411,0.029104,0.000560,0.117481,6.443957e-06,0.000514,0.000045,0.000162,0.024527,0.000029
3,2025-08-12 22:41:43.280,2025-08-12 22:41:47.260,300,499,1,ear_rubbing,0.970215,smoking,0.018251,0.951964,...,0.000279,0.002954,0.000012,0.001749,2.324916e-06,0.006475,0.000017,0.000042,0.018251,0.000003
4,2025-08-12 22:41:45.280,2025-08-12 22:41:49.260,400,599,1,ear_rubbing,0.893137,smoking,0.100547,0.792590,...,0.000060,0.002420,0.000010,0.000704,1.086174e-07,0.003104,0.000008,0.000008,0.100547,0.000001


# Cell 6 — confidence summaries

In [6]:
activity_confidence_summary = (
    window_out
    .groupby("predicted_activity", dropna=False)
    .agg(
        window_count=("predicted_activity", "size"),
        mean_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        min_confidence=("confidence", "min"),
        max_confidence=("confidence", "max"),
        mean_top2_confidence=("top2_confidence", "mean"),
        mean_top1_top2_margin=("top1_top2_margin", "mean"),
        uncertain_windows=("is_uncertain", "sum"),
    )
    .reset_index()
)

activity_confidence_summary["percent_windows"] = (
    100 * activity_confidence_summary["window_count"] / max(len(window_out), 1)
)

band_counts = (
    window_out
    .pivot_table(
        index="predicted_activity",
        columns="confidence_band",
        values="confidence",
        aggfunc="count",
        fill_value=0,
    )
    .reset_index()
)

for band in ["high", "medium", "low"]:
    if band not in band_counts.columns:
        band_counts[band] = 0

activity_confidence_summary = activity_confidence_summary.merge(
    band_counts[["predicted_activity", "high", "medium", "low"]],
    on="predicted_activity",
    how="left",
)

activity_confidence_summary = activity_confidence_summary.sort_values(
    ["window_count", "mean_confidence"],
    ascending=[False, False],
).reset_index(drop=True)

activity_confidence_summary.to_csv(ACTIVITY_CONFIDENCE_CSV, index=False)

print("Saved:", ACTIVITY_CONFIDENCE_CSV)

display(activity_confidence_summary)


class_probability_summary = pd.DataFrame({
    "activity": CLASS_NAMES,
    "mean_probability_all_windows": pred_probs.mean(axis=0),
    "median_probability_all_windows": np.median(pred_probs, axis=0),
    "max_probability_any_window": pred_probs.max(axis=0),
    "windows_where_top1": [(top1_idx == i).sum() for i in range(len(CLASS_NAMES))],
})

class_probability_summary["percent_windows_where_top1"] = (
    100 * class_probability_summary["windows_where_top1"] / max(len(window_out), 1)
)

class_probability_summary = class_probability_summary.sort_values(
    "mean_probability_all_windows",
    ascending=False,
).reset_index(drop=True)

class_probability_summary.to_csv(CLASS_PROB_SUMMARY_CSV, index=False)

print("Saved:", CLASS_PROB_SUMMARY_CSV)

display(class_probability_summary)

Saved: inference_outputs\activity_confidence_summary.csv


,predicted_activity,window_count,mean_confidence,median_confidence,min_confidence,max_confidence,mean_top2_confidence,mean_top1_top2_margin,uncertain_windows,percent_windows,high,medium,low
0,ear_rubbing,511,0.715124,0.728024,0.300169,0.997618,0.193674,0.521450,144,42.023026,186,181,144
1,hand_tapping,478,0.799882,0.846080,0.286041,0.999879,0.156489,0.643392,75,39.309211,270,133,75
2,smoking,202,0.851917,0.937250,0.371445,0.987644,0.099586,0.752331,29,16.611842,152,21,29
3,hair_pulling,17,0.503626,0.459700,0.361625,0.727059,0.286246,0.217381,13,1.398026,0,4,13
4,nail_biting,6,0.523185,0.502161,0.440639,0.657894,0.328447,0.194738,5,0.493421,0,1,5
5,hand_scratching,1,0.435147,0.435147,0.435147,0.435147,0.345014,0.090133,1,0.082237,0,0,1
6,forehead_rubbing,1,0.280442,0.280442,0.280442,0.280442,0.192312,0.088130,1,0.082237,0,0,1


Saved: inference_outputs\class_probability_summary.csv


,activity,mean_probability_all_windows,median_probability_all_windows,max_probability_any_window,windows_where_top1,percent_windows_where_top1
0,hand_tapping,0.383738,0.256620,0.999879,478,39.309211
1,ear_rubbing,0.379793,0.319750,0.997618,511,42.023026
2,smoking,0.178545,0.023192,0.987644,202,16.611842
3,hair_pulling,0.033027,0.005946,0.727059,17,1.398026
4,nail_biting,0.014367,0.001718,0.657894,6,0.493421
5,forehead_rubbing,0.005659,0.000728,0.354605,1,0.082237
6,hand_scratching,0.003497,0.000201,0.435147,1,0.082237
7,sitting,0.001004,0.000258,0.045008,0,0.000000
8,nape_rubbing,0.000223,0.000029,0.016082,0,0.000000
9,standing,0.000129,0.000044,0.008494,0,0.000000


# Cell 7 — row-level soft-vote predictions

In [7]:
def build_row_level_softvote(prod_ds, meta_prod, pred_probs, class_names):
    """
    Projects overlapping window predictions back to row level.

    Each row gets the average probability of all windows that contain it.
    """
    n_rows = len(prod_ds)
    n_classes = len(class_names)

    prob_sum = np.zeros((n_rows, n_classes), dtype=np.float64)
    vote_count = np.zeros(n_rows, dtype=np.int32)

    for i, row in meta_prod.iterrows():
        start_idx = int(row["start_idx"])
        end_idx = int(row["end_idx"])

        start_idx = max(start_idx, 0)
        end_idx = min(end_idx, n_rows - 1)

        if end_idx < start_idx:
            continue

        prob_sum[start_idx:end_idx + 1] += pred_probs[i]
        vote_count[start_idx:end_idx + 1] += 1

    valid = vote_count > 0

    row_probs = np.full((n_rows, n_classes), np.nan, dtype=np.float64)
    row_probs[valid] = prob_sum[valid] / vote_count[valid, None]

    row_pred_idx = np.full(n_rows, -1, dtype=int)
    row_conf = np.full(n_rows, np.nan, dtype=np.float64)

    row_pred_idx[valid] = np.nanargmax(row_probs[valid], axis=1)
    row_conf[valid] = np.nanmax(row_probs[valid], axis=1)

    row_out = prod_ds.copy()

    row_out["predicted_activity"] = None
    row_out.loc[valid, "predicted_activity"] = np.array(class_names)[row_pred_idx[valid]]

    row_out["confidence"] = row_conf
    row_out["overlap_vote_count"] = vote_count

    for i, cls in enumerate(class_names):
        row_out[f"prob_{cls}"] = row_probs[:, i]

    row_out["confidence_band"] = np.where(
        row_out["confidence"] >= HIGH_CONF_THRESHOLD,
        "high",
        np.where(
            row_out["confidence"] >= MEDIUM_CONF_THRESHOLD,
            "medium",
            "low",
        ),
    )

    row_out.loc[~valid, "confidence_band"] = None

    row_out["is_uncertain"] = row_out["confidence"] < UNCERTAIN_THRESHOLD
    row_out.loc[~valid, "is_uncertain"] = True

    return row_out


row_out = build_row_level_softvote(
    prod_ds=prod_ds,
    meta_prod=meta_prod,
    pred_probs=pred_probs,
    class_names=CLASS_NAMES,
)

row_out.to_csv(ROW_PRED_CSV, index=False)

print("Saved:", ROW_PRED_CSV)
print("Rows:", row_out.shape)

row_out.head()

Saved: inference_outputs\production_row_predictions.csv
Rows: (121726, 23)


,timestamp,Ax_w,Ay_w,Az_w,Gx_w,Gy_w,Gz_w,predicted_activity,confidence,overlap_vote_count,...,prob_hand_scratching,prob_hand_tapping,prob_knuckles_cracking,prob_nail_biting,prob_nape_rubbing,prob_sitting,prob_smoking,prob_standing,confidence_band,is_uncertain
0,2025-08-12 22:41:37.240,-1.363688,-6.280026,-11.154101,-34.847054,0.172556,16.718046,ear_rubbing,0.575845,1,...,0.000044,0.305295,0.000002,0.001615,0.000004,0.000287,0.062697,0.000364,low,True
1,2025-08-12 22:41:37.260,-0.793505,-5.509558,-11.386176,-23.874311,11.460101,12.290460,ear_rubbing,0.575845,1,...,0.000044,0.305295,0.000002,0.001615,0.000004,0.000287,0.062697,0.000364,low,True
2,2025-08-12 22:41:37.280,-0.479904,-5.306305,-9.510236,26.123428,16.640148,-6.189626,ear_rubbing,0.575845,1,...,0.000044,0.305295,0.000002,0.001615,0.000004,0.000287,0.062697,0.000364,low,True
3,2025-08-12 22:41:37.300,-1.948126,-4.772177,-7.411892,32.581009,21.680162,-20.539654,ear_rubbing,0.575845,1,...,0.000044,0.305295,0.000002,0.001615,0.000004,0.000287,0.062697,0.000364,low,True
4,2025-08-12 22:41:37.320,-3.040977,-1.988092,-3.016974,33.561008,32.565163,-19.349654,ear_rubbing,0.575845,1,...,0.000044,0.305295,0.000002,0.001615,0.000004,0.000287,0.062697,0.000364,low,True


# Cell 8 — optional reference agreement check

In [8]:
def normalize_activity_name(x):
    return str(x).strip().lower().replace(" ", "_").replace("-", "_")


if y_ref is not None and len(y_ref) > 0 and X_ref.shape[0] > 0:
    print("Checking model agreement on reference data.")
    print("NOTE: This is not test accuracy unless the reference CSV is held-out.")

    ref_probs = model.predict(X_ref, verbose=0, batch_size=256)
    ref_probs = np.asarray(ref_probs)

    ref_pred_idx = ref_probs.argmax(axis=1)
    ref_pred_label = classes[ref_pred_idx]

    y_true_norm = np.array([normalize_activity_name(v) for v in y_ref])
    y_pred_norm = np.array([normalize_activity_name(v) for v in ref_pred_label])

    ref_agreement = np.mean(y_true_norm == y_pred_norm)

    print("Reference-data agreement:", round(float(ref_agreement), 4))

    try:
        from sklearn.metrics import classification_report

        print(
            classification_report(
                y_true_norm,
                y_pred_norm,
                labels=CLASS_NAMES,
                zero_division=0,
            )
        )

    except Exception as e:
        print("Could not print classification report:", e)

else:
    print("Reference agreement skipped because no reference labels/windows were loaded.")

Checking model agreement on reference data.
NOTE: This is not test accuracy unless the reference CSV is held-out.
Reference-data agreement: 0.1029
                   precision    recall  f1-score   support

      ear_rubbing       0.11      0.53      0.18       318
 forehead_rubbing       0.00      0.00      0.00       319
     hair_pulling       0.20      0.01      0.01       317
  hand_scratching       0.00      0.00      0.00       303
     hand_tapping       0.11      0.60      0.18       303
knuckles_cracking       0.00      0.00      0.00       315
      nail_biting       0.00      0.00      0.00       342
     nape_rubbing       0.00      0.00      0.00       293
          sitting       0.00      0.00      0.00       312
          smoking       0.01      0.01      0.01       322
         standing       0.00      0.00      0.00       305

         accuracy                           0.10      3449
        macro avg       0.04      0.10      0.04      3449
     weighted avg       0

# Cell 9 — feature drift functions

In [9]:
def add_magnitude_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["accel_mag"] = np.sqrt(
        out["Ax_w"] ** 2 + out["Ay_w"] ** 2 + out["Az_w"] ** 2
    )

    out["gyro_mag"] = np.sqrt(
        out["Gx_w"] ** 2 + out["Gy_w"] ** 2 + out["Gz_w"] ** 2
    )

    return out


def _psi_1d(reference_vals: np.ndarray, production_vals: np.ndarray, bins: int = 10) -> float:
    reference_vals = np.asarray(reference_vals, dtype=float)
    production_vals = np.asarray(production_vals, dtype=float)

    reference_vals = reference_vals[np.isfinite(reference_vals)]
    production_vals = production_vals[np.isfinite(production_vals)]

    if len(reference_vals) == 0 or len(production_vals) == 0:
        return np.nan

    edges = np.quantile(reference_vals, np.linspace(0, 1, bins + 1))
    edges = np.unique(edges)

    if len(edges) < 3:
        return 0.0

    edges[0] = -np.inf
    edges[-1] = np.inf

    ref_hist, _ = np.histogram(reference_vals, bins=edges)
    prod_hist, _ = np.histogram(production_vals, bins=edges)

    ref_pct = ref_hist / max(ref_hist.sum(), 1)
    prod_pct = prod_hist / max(prod_hist.sum(), 1)

    eps = 1e-8
    ref_pct = np.clip(ref_pct, eps, None)
    prod_pct = np.clip(prod_pct, eps, None)

    psi = np.sum((prod_pct - ref_pct) * np.log(prod_pct / ref_pct))

    return float(psi)


def _ks_1d(reference_vals: np.ndarray, production_vals: np.ndarray) -> float:
    reference_vals = np.asarray(reference_vals, dtype=float)
    production_vals = np.asarray(production_vals, dtype=float)

    reference_vals = np.sort(reference_vals[np.isfinite(reference_vals)])
    production_vals = np.sort(production_vals[np.isfinite(production_vals)])

    if len(reference_vals) == 0 or len(production_vals) == 0:
        return np.nan

    x = np.sort(np.unique(np.concatenate([reference_vals, production_vals])))

    cdf_ref = np.searchsorted(reference_vals, x, side="right") / len(reference_vals)
    cdf_prod = np.searchsorted(production_vals, x, side="right") / len(production_vals)

    return float(np.max(np.abs(cdf_ref - cdf_prod)))


def compute_feature_drift(reference_df: pd.DataFrame, production_df: pd.DataFrame, feature_cols):
    if reference_df is None or len(reference_df) == 0:
        raise ValueError("Reference data is empty.")

    if production_df is None or len(production_df) == 0:
        raise ValueError("Production data is empty.")

    rows = []

    for col in feature_cols:
        ref = pd.to_numeric(reference_df[col], errors="coerce").dropna().to_numpy()
        prod = pd.to_numeric(production_df[col], errors="coerce").dropna().to_numpy()

        psi = _psi_1d(ref, prod)
        ks = _ks_1d(ref, prod)

        ref_mean = float(np.mean(ref)) if len(ref) else np.nan
        prod_mean = float(np.mean(prod)) if len(prod) else np.nan

        ref_std = float(np.std(ref)) if len(ref) else np.nan
        prod_std = float(np.std(prod)) if len(prod) else np.nan

        if not np.isfinite(ref_std) or ref_std == 0:
            mean_shift_z = np.nan
            std_ratio = np.nan
        else:
            mean_shift_z = abs(prod_mean - ref_mean) / ref_std
            std_ratio = prod_std / ref_std

        is_drift = (
            (np.isfinite(psi) and psi > DRIFT_PSI_THRESHOLD)
            or (np.isfinite(ks) and ks > DRIFT_KS_THRESHOLD)
            or (np.isfinite(mean_shift_z) and mean_shift_z > DRIFT_MEAN_SHIFT_Z_THRESHOLD)
            or (np.isfinite(std_ratio) and std_ratio < DRIFT_STD_RATIO_LOW)
            or (np.isfinite(std_ratio) and std_ratio > DRIFT_STD_RATIO_HIGH)
        )

        rows.append({
            "feature": col,
            "psi": psi,
            "ks_stat": ks,
            "reference_mean": ref_mean,
            "production_mean": prod_mean,
            "mean_shift_z": mean_shift_z,
            "reference_std": ref_std,
            "production_std": prod_std,
            "std_ratio": std_ratio,
            "is_drift": bool(is_drift),
        })

    drift_df = pd.DataFrame(rows).sort_values(
        ["is_drift", "psi", "ks_stat"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    psi_norm = np.clip(drift_df["psi"].fillna(0).mean() / DRIFT_PSI_THRESHOLD, 0, 1)
    ks_norm = np.clip(drift_df["ks_stat"].fillna(0).mean() / DRIFT_KS_THRESHOLD, 0, 1)
    mean_norm = np.clip(
        drift_df["mean_shift_z"].fillna(0).mean() / DRIFT_MEAN_SHIFT_Z_THRESHOLD,
        0,
        1,
    )
    drift_rate = drift_df["is_drift"].mean()

    overall_score = float(np.mean([psi_norm, ks_norm, mean_norm, drift_rate]))

    return drift_df, overall_score

# Cell 10 — activity distribution drift functions

In [10]:
def compute_activity_distribution_drift(reference_labels, production_pred_labels, class_names):
    ref = pd.Series(reference_labels, dtype="object").map(normalize_activity_name)
    prod = pd.Series(production_pred_labels, dtype="object").map(normalize_activity_name)

    ref_counts = ref.value_counts().reindex(class_names, fill_value=0)
    prod_counts = prod.value_counts().reindex(class_names, fill_value=0)

    ref_pct = ref_counts / max(ref_counts.sum(), 1)
    prod_pct = prod_counts / max(prod_counts.sum(), 1)

    eps = 1e-8

    psi_terms = (
        (prod_pct.clip(eps) - ref_pct.clip(eps))
        * np.log(prod_pct.clip(eps) / ref_pct.clip(eps))
    )

    total_variation_distance = 0.5 * np.abs(prod_pct - ref_pct).sum()

    out = pd.DataFrame({
        "activity": class_names,
        "reference_count": ref_counts.values,
        "production_count": prod_counts.values,
        "reference_pct": ref_pct.values,
        "production_pct": prod_pct.values,
        "pct_difference": (prod_pct - ref_pct).values,
        "psi_component": psi_terms.values,
    })

    overall_activity_psi = float(psi_terms.sum())

    is_activity_distribution_drift = bool(
        total_variation_distance > ACTIVITY_DIST_DRIFT_TV_THRESHOLD
    )

    return (
        out,
        overall_activity_psi,
        float(total_variation_distance),
        is_activity_distribution_drift,
    )

# Cell 11 — run drift reports

In [11]:
# Add magnitude features for more useful drift analysis.
reference_drift_ds = add_magnitude_features(reference_ds) if reference_ds is not None else None
prod_drift_ds = add_magnitude_features(prod_ds)

DRIFT_FEATURE_COLS = FEATURE_COLS + ["accel_mag", "gyro_mag"]


# -----------------
# FEATURE DRIFT
# -----------------

if reference_drift_ds is None or len(reference_drift_ds) == 0:
    print("Skipping feature drift: no reference data loaded.")

    feature_drift_df = pd.DataFrame()
    overall_feature_drift_score = np.nan

else:
    feature_drift_df, overall_feature_drift_score = compute_feature_drift(
        reference_df=reference_drift_ds,
        production_df=prod_drift_ds,
        feature_cols=DRIFT_FEATURE_COLS,
    )

    feature_drift_df.to_csv(FEATURE_DRIFT_CSV, index=False)

    print("Saved:", FEATURE_DRIFT_CSV)
    print("Overall feature drift score:", round(overall_feature_drift_score, 4))
    print(
        "Drifted features:",
        int(feature_drift_df["is_drift"].sum()),
        "/",
        len(feature_drift_df),
    )


# -----------------
# ACTIVITY DISTRIBUTION DRIFT
# -----------------

if y_ref is None or len(y_ref) == 0:
    print("Skipping activity distribution drift: no reference labels.")

    activity_drift_df = pd.DataFrame()
    activity_distribution_psi = np.nan
    activity_tv_distance = np.nan
    is_activity_distribution_drift = False

else:
    (
        activity_drift_df,
        activity_distribution_psi,
        activity_tv_distance,
        is_activity_distribution_drift,
    ) = compute_activity_distribution_drift(
        reference_labels=y_ref,
        production_pred_labels=window_out["predicted_activity"],
        class_names=CLASS_NAMES,
    )

    activity_drift_df.to_csv(ACTIVITY_DRIFT_CSV, index=False)

    print("Saved:", ACTIVITY_DRIFT_CSV)
    print("Activity distribution PSI:", round(activity_distribution_psi, 4))
    print("Activity TV distance:", round(activity_tv_distance, 4))
    print("Activity distribution drift:", is_activity_distribution_drift)


# -----------------
# DRIFT SUMMARY
# -----------------

drift_summary = pd.DataFrame([
    {
        "metric": "feature_drift_overall_score",
        "value": overall_feature_drift_score,
        "threshold": "closer to 1 means higher drift",
        "is_drift": bool(feature_drift_df["is_drift"].any()) if len(feature_drift_df) else False,
    },
    {
        "metric": "feature_drifted_feature_count",
        "value": int(feature_drift_df["is_drift"].sum()) if len(feature_drift_df) else 0,
        "threshold": (
            f"PSI>{DRIFT_PSI_THRESHOLD}, "
            f"KS>{DRIFT_KS_THRESHOLD}, "
            f"mean shift>{DRIFT_MEAN_SHIFT_Z_THRESHOLD}, "
            f"std ratio outside [{DRIFT_STD_RATIO_LOW}, {DRIFT_STD_RATIO_HIGH}]"
        ),
        "is_drift": bool(feature_drift_df["is_drift"].any()) if len(feature_drift_df) else False,
    },
    {
        "metric": "activity_distribution_psi",
        "value": activity_distribution_psi,
        "threshold": "higher means class distribution differs more",
        "is_drift": bool(is_activity_distribution_drift),
    },
    {
        "metric": "activity_distribution_total_variation_distance",
        "value": activity_tv_distance,
        "threshold": f">{ACTIVITY_DIST_DRIFT_TV_THRESHOLD}",
        "is_drift": bool(is_activity_distribution_drift),
    },
])

drift_summary.to_csv(DRIFT_SUMMARY_CSV, index=False)

print("Saved:", DRIFT_SUMMARY_CSV)

print("\nFeature drift:")
display(feature_drift_df)

print("\nActivity distribution drift:")
display(activity_drift_df)

print("\nDrift summary:")
display(drift_summary)

Saved: inference_outputs\feature_drift_report.csv
Overall feature drift score: 0.7154
Drifted features: 6 / 8
Saved: inference_outputs\activity_distribution_drift.csv
Activity distribution PSI: 7.9485
Activity TV distance: 0.706
Activity distribution drift: True
Saved: inference_outputs\drift_summary.csv

Feature drift:


,feature,psi,ks_stat,reference_mean,production_mean,mean_shift_z,reference_std,production_std,std_ratio,is_drift
0,Ay_w,1.490319,0.459701,1.282084,5.813709,1.041402,4.351464,3.763802,0.864951,True
1,Ax_w,0.834913,0.352066,3.218580,0.413659,0.427037,6.568332,5.184223,0.789275,True
2,Az_w,0.630754,0.282830,-3.528949,-1.582712,0.601402,3.236165,4.881660,1.508471,True
3,Gy_w,0.213098,0.093634,0.225205,1.017890,0.053517,14.811710,35.948045,2.427002,True
4,Gz_w,0.172826,0.072842,0.088668,-0.211994,0.021223,14.166767,37.047113,2.615072,True
5,accel_mag,0.169751,0.087531,9.828424,9.939977,0.174315,0.639951,1.595349,2.492924,True
6,gyro_mag,0.242905,0.126646,28.112291,43.776498,0.339946,46.078480,68.580383,1.488339,False
7,Gx_w,0.052896,0.036206,0.599312,0.000767,0.011988,49.930206,62.879402,1.259346,False



Activity distribution drift:


,activity,reference_count,production_count,reference_pct,production_pct,pct_difference,psi_component
0,ear_rubbing,318,511,0.092201,0.420230,0.328030,0.497567
1,forehead_rubbing,319,1,0.092491,0.000822,-0.091668,0.432919
2,hair_pulling,317,17,0.091911,0.013980,-0.077930,0.146756
3,hand_scratching,303,1,0.087852,0.000822,-0.087029,0.406532
4,hand_tapping,303,478,0.087852,0.393092,0.305241,0.457371
5,knuckles_cracking,315,0,0.091331,0.000000,-0.091331,1.463797
6,nail_biting,342,6,0.099159,0.004934,-0.094225,0.282725
7,nape_rubbing,293,0,0.084952,0.000000,-0.084952,1.355413
8,sitting,312,0,0.090461,0.000000,-0.090461,1.448990
9,smoking,322,202,0.093360,0.166118,0.072758,0.041926



Drift summary:


,metric,value,threshold,is_drift
0,feature_drift_overall_score,0.715397,closer to 1 means higher drift,True
1,feature_drifted_feature_count,6.000000,"PSI>0.25, KS>0.2, mean shift>2.0, std ratio ou...",True
2,activity_distribution_psi,7.948470,higher means class distribution differs more,True
3,activity_distribution_total_variation_distance,0.706028,>0.25,True
